# MOD-08 · NB-03 — Streamlit Dashboards for Clinical Analytics
### Health Analytics with Python · Module 08: Reproducible Research & Deployment
---
**Learning objectives**
- Build a multi-tab clinical analytics Streamlit dashboard
- Use `st.cache_data` and `st.session_state` for performance
- Create interactive filters, KPI metrics, and downloadable tables
- Implement patient risk alerting with adjustable thresholds
- Apply authentication and multi-page navigation patterns
- Deploy Streamlit apps via Docker and Streamlit Community Cloud

**Estimated time:** 3 hours | **Level:** Intermediate | **Libraries:** `streamlit`, `matplotlib`, `sklearn`

## 1. Setup & Synthetic Dashboard Dataset

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
os.makedirs("/tmp/mod08", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white"})
np.random.seed(42); N = 3000
def sigmoid(x): return 1/(1+np.exp(-x))
df = pd.DataFrame({
    "patient_id" : [f"PT-{i:05d}" for i in range(N)],
    "age"        : np.random.normal(60,15,N).clip(18,90).astype(int),
    "sex"        : np.random.choice(["M","F"],N,p=[0.48,0.52]),
    "payer"      : np.random.choice(["Medicare","Medicaid","Private","Self-pay"],N,p=[0.40,0.22,0.28,0.10]),
    "unit"       : np.random.choice(["Cardiology","Pulmonology","Medicine","Surgery"],N),
    "los_days"   : np.random.gamma(2.5,1.8,N).clip(1,30).astype(int),
    "diabetes"   : np.random.binomial(1, 0.35, N),
    "hf"         : np.random.binomial(1, 0.22, N),
    "copd"       : np.random.binomial(1, 0.18, N),
})
logit = -3.2+0.6*df.hf+0.5*df.diabetes+0.02*(df.age-60)+0.2*(df.los_days>7).astype(int)
df["readmit_prob"] = sigmoid(logit.values)
df["readmitted"]   = np.random.binomial(1, df.readmit_prob.values, N)
df["admit_date"]   = pd.date_range("2023-01-01", periods=N, freq="6h")[:N]
print(f"Dashboard dataset: {df.shape} | Readmission: {df.readmitted.mean()*100:.1f}%")


## 2. Streamlit Architecture for Clinical Dashboards

```
app.py                    <- Entry point
pages/
  01_overview.py          <- Aggregate KPIs and trends
  02_patient_risk.py      <- Individual risk scores
  03_model_performance.py <- AUC, calibration, DCA
  04_documentation.py     <- Methods and data dictionary
src/
  model.py                <- load_model(), predict()
  data.py                 <- load_data(), filter_data()
  plots.py                <- reusable plot functions
```

**Key Streamlit concepts for clinical apps:**
- `st.cache_data`: cache expensive data loads (refreshes every `ttl` seconds)
- `st.session_state`: persist user selections across pages
- `st.sidebar`: filters that update all tabs simultaneously
- `st.metrics`: KPI cards with delta indicators
- `st.download_button`: export high-risk patient lists to CSV


## 3. Complete app.py

In [ ]:
STREAMLIT_APP = (
    "# app.py -- Clinical Readmission Dashboard\n"
    "import streamlit as st\n"
    "import pandas as pd\n"
    "import numpy as np\n"
    "import matplotlib.pyplot as plt\n"
    "from sklearn.linear_model import LogisticRegression\n"
    "from sklearn.metrics import roc_auc_score, roc_curve\n\n"
    "# Page configuration\n"
    "st.set_page_config(\n"
    "    page_title='Readmission Analytics Dashboard',\n"
    "    page_icon='🏥',\n"
    "    layout='wide',\n"
    "    initial_sidebar_state='expanded'\n"
    ")\n\n"
    "# ── Sidebar (filters) ──────────────────────────────────────────────\n"
    "st.sidebar.title('Filters')\n"
    "st.sidebar.markdown('---')\n"
    "selected_units = st.sidebar.multiselect(\n"
    "    'Hospital Unit',\n"
    "    options=['Cardiology','Pulmonology','Medicine','Surgery'],\n"
    "    default=['Cardiology','Pulmonology','Medicine','Surgery']\n"
    ")\n"
    "age_range = st.sidebar.slider('Age Range', 18, 90, (30, 80))\n"
    "risk_threshold = st.sidebar.slider(\n"
    "    'Risk Alert Threshold', 0.10, 0.50, 0.25,\n"
    "    help='Flag patients above this predicted readmission probability'\n"
    ")\n\n"
    "# ── Load and filter data ───────────────────────────────────────────\n"
    "@st.cache_data\n"
    "def load_data():\n"
    "    return pd.read_parquet('data/processed/features.parquet')\n\n"
    "df = load_data()\n"
    "df_filtered = df[\n"
    "    df.unit.isin(selected_units) &\n"
    "    df.age.between(*age_range)\n"
    "]\n\n"
    "# ── KPI Header ─────────────────────────────────────────────────────\n"
    "st.title('30-Day Readmission Analytics Dashboard')\n"
    "col1, col2, col3, col4 = st.columns(4)\n"
    "col1.metric('Total Patients', f'{len(df_filtered):,}')\n"
    "col2.metric('Readmission Rate', f"{df_filtered.readmitted.mean()*100:.1f}%",\n"
    "            delta=f"{df_filtered.readmitted.mean()*100 - df.readmitted.mean()*100:.1f}pp vs overall")\n"
    "col3.metric('Mean LOS (days)', f"{df_filtered.los_days.mean():.1f}")\n"
    "col4.metric('High-Risk Patients',\n"
    "            f"{(df_filtered.readmit_prob >= risk_threshold).sum():,}")\n\n"
    "# ── Tabs layout ────────────────────────────────────────────────────\n"
    "tab1, tab2, tab3 = st.tabs(['Overview', 'Patient Risk List', 'Model Performance'])\n\n"
    "with tab1:\n"
    "    c1, c2 = st.columns(2)\n"
    "    with c1:\n"
    "        st.subheader('Readmission Rate by Unit')\n"
    "        unit_rates = df_filtered.groupby('unit')['readmitted'].mean().sort_values()*100\n"
    "        st.bar_chart(unit_rates)\n"
    "    with c2:\n"
    "        st.subheader('Risk Score Distribution')\n"
    "        st.line_chart(df_filtered['readmit_prob'].value_counts(bins=20).sort_index())\n\n"
    "with tab2:\n"
    "    st.subheader(f'High-Risk Patients (probability >= {risk_threshold:.0%})')\n"
    "    high_risk = df_filtered[df_filtered.readmit_prob >= risk_threshold]\n"
    "    st.dataframe(\n"
    "        high_risk[['patient_id','age','unit','los_days','readmit_prob']]\n"
    "        .sort_values('readmit_prob', ascending=False),\n"
    "        use_container_width=True\n"
    "    )\n"
    "    csv = high_risk.to_csv(index=False)\n"
    "    st.download_button('Download High-Risk List', csv, 'high_risk_patients.csv')\n\n"
    "with tab3:\n"
    "    st.subheader('Model Performance Metrics')\n"
    "    auc = roc_auc_score(df_filtered.readmitted, df_filtered.readmit_prob)\n"
    "    st.metric('AUC-ROC', f'{auc:.3f}')\n"
)

print("Streamlit application code (app.py):")
print(STREAMLIT_APP[:1800])
app_path = Path("/tmp/mod08/app.py")
app_path.write_text(STREAMLIT_APP)
print(f"\nApp saved to: {app_path}")
print("\nTo run: streamlit run /tmp/mod08/app.py")


## 4. Dashboard Preview (matplotlib simulation)

In [ ]:
# Demonstrate Streamlit component patterns without running the server
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df[["age","los_days","diabetes","hf","copd"]]
y = df["readmitted"]
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
lr = LogisticRegression(C=1,max_iter=500,class_weight="balanced",random_state=42)
lr.fit(X_tr,y_tr)
df["readmit_prob"] = lr.predict_proba(df[X.columns])[:,1]
prob_te = lr.predict_proba(X_te)[:,1]
auc = roc_auc_score(y_te, prob_te)

fig = plt.figure(figsize=(18,10))
fig.suptitle("Streamlit Dashboard Preview: Clinical Readmission Analytics", fontsize=13, y=1.01)
gs = gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.35)

# KPI metrics (simulated as text boxes)
ax_kpi = fig.add_subplot(gs[0,:])
ax_kpi.axis("off")
kpi_data = [
    ("Total Patients", f"{len(df):,}", "#4878CF"),
    ("Readmission Rate", f"{df.readmitted.mean()*100:.1f}%", "#D65F5F"),
    ("Mean LOS", f"{df.los_days.mean():.1f} days", "#6ACC65"),
    ("High Risk (>25%)", f"{(df.readmit_prob>0.25).sum():,}", "#D4A017"),
    ("Model AUC", f"{auc:.3f}", "#B47CC7"),
]
for i,(label,value,color) in enumerate(kpi_data):
    x = 0.05 + i*0.19
    rect = plt.Rectangle((x,0.1),0.17,0.8,fill=True,facecolor=color,alpha=0.15,
                           edgecolor=color,linewidth=2,transform=ax_kpi.transAxes)
    ax_kpi.add_patch(rect)
    ax_kpi.text(x+0.085,0.65,value,ha="center",va="center",fontsize=16,fontweight="bold",
                color=color,transform=ax_kpi.transAxes)
    ax_kpi.text(x+0.085,0.25,label,ha="center",va="center",fontsize=9,color="gray",
                transform=ax_kpi.transAxes)
ax_kpi.set_title("KPI Header Row",fontsize=10,pad=8,loc="left")

# Unit rates
ax_a = fig.add_subplot(gs[1,0])
unit_r = df.groupby("unit")["readmitted"].mean().sort_values()*100
colors_unit = ["#D65F5F" if r>df.readmitted.mean()*100 else "#4878CF" for r in unit_r.values]
bars = ax_a.bar(unit_r.index,unit_r.values,color=colors_unit,alpha=0.85,edgecolor="white")
ax_a.axhline(df.readmitted.mean()*100,color="black",ls="--",lw=1.5,label="Overall avg")
ax_a.set_ylabel("Readmission rate (%)"); ax_a.set_title("Tab 1: Rate by Unit")
ax_a.tick_params(axis="x",rotation=20); ax_a.legend(fontsize=8)
ax_a.spines["top"].set_visible(False); ax_a.spines["right"].set_visible(False)

# Risk distribution
ax_b = fig.add_subplot(gs[1,1])
for val,color,lbl in [(0,"#4878CF","Not readmitted"),(1,"#D65F5F","Readmitted")]:
    ax_b.hist(df[df.readmitted==val]["readmit_prob"],bins=25,alpha=0.6,
              color=color,label=lbl,density=True)
ax_b.axvline(0.25,color="black",ls="--",lw=1.5,label="Alert threshold (25%)")
ax_b.set_xlabel("Predicted probability"); ax_b.set_ylabel("Density")
ax_b.set_title("Tab 1: Risk Score Distribution"); ax_b.legend(fontsize=7)
ax_b.spines["top"].set_visible(False); ax_b.spines["right"].set_visible(False)

# ROC curve
ax_c = fig.add_subplot(gs[1,2])
fpr,tpr,_ = roc_curve(y_te,prob_te)
ax_c.plot(fpr,tpr,color="#D65F5F",lw=2.5,label=f"AUC={auc:.3f}")
ax_c.plot([0,1],[0,1],"k--",lw=1); ax_c.fill_between(fpr,tpr,alpha=0.1,color="#D65F5F")
ax_c.set_xlabel("1-Specificity"); ax_c.set_ylabel("Sensitivity")
ax_c.set_title("Tab 3: Model Performance"); ax_c.legend(fontsize=9)
ax_c.spines["top"].set_visible(False); ax_c.spines["right"].set_visible(False)

plt.savefig("/tmp/mod08/streamlit_preview.png",bbox_inches="tight",dpi=150)
plt.show()
print("Dashboard preview figure saved.")


## 5. Advanced Streamlit Patterns

In [ ]:
# Advanced Streamlit patterns for clinical dashboards
ADVANCED_PATTERNS = (
    "# Pattern 1: Caching for expensive computations\n"
    "@st.cache_data(ttl=3600)  # Re-run after 1 hour\n"
    "def run_model(features_path, model_path):\n"
    "    df = pd.read_parquet(features_path)\n"
    "    model = pickle.load(open(model_path,'rb'))\n"
    "    return model.predict_proba(df[FEATURES])[:,1]\n\n"
    "# Pattern 2: Session state for multi-page apps\n"
    "if 'selected_patient' not in st.session_state:\n"
    "    st.session_state.selected_patient = None\n\n"
    "# Pattern 3: Real-time updates\n"
    "placeholder = st.empty()\n"
    "for i in range(10):\n"
    "    with placeholder.container():\n"
    "        st.metric('Patients Processed', i*100)\n"
    "        time.sleep(0.5)\n\n"
    "# Pattern 4: Plotly for interactive charts\n"
    "import plotly.express as px\n"
    "fig = px.scatter(df, x='age', y='readmit_prob', color='unit',\n"
    "                  hover_data=['patient_id','los_days'],\n"
    "                  title='Risk by Age and Unit')\n"
    "st.plotly_chart(fig, use_container_width=True)\n\n"
    "# Pattern 5: Authentication\n"
    "# pip install streamlit-authenticator\n"
    "import streamlit_authenticator as stauth\n"
    "authenticator = stauth.Authenticate(credentials, cookie_name, key, expiry_days)\n"
    "name, status, username = authenticator.login('Login', 'main')\n"
    "if status: st.write(f'Welcome, {name}!')\n"
    "elif status == False: st.error('Username or password incorrect')\n\n"
    "# Pattern 6: Multi-page navigation\n"
    "# pages/\n"
    "#   01_overview.py    <- st.set_page_config() first\n"
    "#   02_patients.py\n"
    "#   03_model.py\n"
    "# streamlit run pages/01_overview.py\n"
)
print("Advanced Streamlit Patterns:")
print(ADVANCED_PATTERNS)


## 6. Deployment Options

In [ ]:
# Deployment configurations for Streamlit apps
DEPLOY_OPTIONS = {
    "Streamlit Community Cloud": {
        "requirements": "requirements.txt in repo root",
        "steps": [
            "1. Push code to GitHub",
            "2. Visit share.streamlit.io",
            "3. Connect GitHub repo",
            "4. Specify app.py path",
            "5. Click Deploy"
        ],
        "cost": "Free for public apps",
        "best_for": "Demo/teaching apps",
    },
    "Docker + AWS/GCP/Azure": {
        "dockerfile": (
            "FROM python:3.10-slim\n"
            "WORKDIR /app\n"
            "COPY requirements.txt .\n"
            "RUN pip install -r requirements.txt\n"
            "COPY . .\n"
            "EXPOSE 8501\n"
            "CMD ['streamlit','run','app.py',\n"
            "     '--server.port=8501',\n"
            "     '--server.address=0.0.0.0']\n"
        ),
        "cost": "Variable (~$50-200/month)",
        "best_for": "Production clinical use",
    },
    "Kubernetes (enterprise)": {
        "best_for": "High-availability clinical systems",
        "features": ["Auto-scaling","Health checks","Rolling updates","Secrets management"],
    }
}

for platform, config in DEPLOY_OPTIONS.items():
    print(f"\n{'='*50}")
    print(f"Platform: {platform}")
    if "steps" in config:
        print("Steps:")
        for step in config["steps"]: print(f"  {step}")
    if "dockerfile" in config:
        print("Dockerfile:")
        print(config["dockerfile"])
    print(f"Best for: {config.get('best_for','')}")
    if "cost" in config: print(f"Cost: {config['cost']}")


## 7. Knowledge Check
1. `st.cache_data` with `ttl=3600`. Your model is re-trained daily. How do you ensure the dashboard uses the latest model?
2. A clinician wants to sort the high-risk patient list by `los_days`. How do you make the table column-sortable?
3. Your Streamlit app uses PHI (patient names, MRNs). What security measures are needed before deployment?
4. The dashboard loads 50,000 patients and is slow. Describe three strategies to improve performance.
5. Build a clinical alert system: show a red badge in the sidebar showing the count of patients above the risk threshold.


In [ ]:
# Exercise 5 — Alert count simulation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

RISK_THRESHOLD = 0.25
high_risk_count = (df.readmit_prob >= RISK_THRESHOLD).sum()
high_risk_pct   = high_risk_count/len(df)*100

# Simulate sidebar with alert badge
fig, ax = plt.subplots(figsize=(4,3))
ax.axis("off")
ax.set_facecolor("#f8f9fa")
# Alert badge
circle = plt.Circle((0.5,0.6),0.25,color="#D65F5F",zorder=3)
ax.add_patch(circle)
ax.text(0.5,0.6,str(high_risk_count),ha="center",va="center",fontsize=20,
        fontweight="bold",color="white",zorder=4,transform=ax.transAxes)
ax.text(0.5,0.25,"High Risk Alerts",ha="center",fontsize=11,
        color="#333",transform=ax.transAxes)
ax.text(0.5,0.12,f"(>{RISK_THRESHOLD:.0%} predicted readmission)",
        ha="center",fontsize=8,color="gray",transform=ax.transAxes)
ax.set_title("Streamlit Sidebar Alert Badge",fontsize=10)
plt.tight_layout(); plt.show()
print(f"High-risk patients (>{RISK_THRESHOLD:.0%}): {high_risk_count} ({high_risk_pct:.1f}%)")


---
**Next → NB-04: MLflow Model Tracking & Registry**